## 03. Feature Engineering: Classical Representation

U ovom notebooku proteinske sekvence se transformišu u numeričke reprezentacije
pogodne za primjenu klasičnih metoda mašinskog učenja.

Razmatrane su tri grupe obilježja:

1. **AAC-CV (Amino Acid Composition, count vectorizer)** — frekvencija 20 standardnih aminokiselina,
2. **TF-IDF** — karakter-n-gram reprezentacija proteinskih sekvenci,
3. **Physicochemical Features (PH)** — fizičko-hemijske osobine proteinskih sekvenci.


### Uvoz biblioteka

In [1]:
import os
import ast
import joblib
import numpy as np
import pandas as pd

from Bio.SeqUtils.ProtParam import ProteinAnalysis

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, Normalizer, MultiLabelBinarizer
from sklearn.feature_extraction.text import CountVectorizer

### Učitavanje podataka 

In [2]:
DATA_DIR = "../../data/processed/localization"
FEATURES_DIR = "../../data/features/localization"
os.makedirs(FEATURES_DIR, exist_ok=True)

DATASET_PATH = os.path.join(DATA_DIR, "dataset_final.csv")
PHYSCHEM_PATH = os.path.join(DATA_DIR, "physicochemical_features.csv")

In [3]:
dataset = pd.read_csv(DATASET_PATH)

dataset['label_list'] = dataset['label'].apply(ast.literal_eval)

print(f"Ukupan broj proteina: {len(dataset)}")
dataset.head()

Ukupan broj proteina: 14566


,Sequence,Entry,label,num_classes,is_multilabel,Cell membrane,Cytoplasm,Mitochondrion,Nucleus,Secreted,label_list
0,AEYFQHWGQGTLVTVSS,A0A0C4DH62,"['Cell membrane', 'Secreted']",2,True,1,0,0,0,1,"[Cell membrane, Secreted]"
1,AKNIQYFGAGTRLSVL,A0A0A0MT87,['Cell membrane'],1,False,1,0,0,0,0,[Cell membrane]
2,APTKAPDVFPIISGCRHPKDNSPVVLACLITGYHPTSVTVTWYMGT...,P01880,"['Cell membrane', 'Secreted']",2,True,1,0,0,0,1,"[Cell membrane, Secreted]"
3,ASPTSPKVFPLSLCSTQPDGNVVIACLVQGFFPQEPLSVTWSESGQ...,P01876,"['Cell membrane', 'Secreted']",2,True,1,0,0,0,1,"[Cell membrane, Secreted]"
4,ASPTSPKVFPLSLDSTPQDGNVVVACLVQGFFPQEPLSVTWSESGQ...,P01877,"['Cell membrane', 'Secreted']",2,True,1,0,0,0,1,"[Cell membrane, Secreted]"


### Fizičko-hemijske osobine

In [4]:
def extract_physicochemical(sequence):
    analysed_seq = ProteinAnalysis(sequence)
    return {
        "MW": analysed_seq.molecular_weight(),
        "pI": analysed_seq.isoelectric_point(),
        "GRAVY": analysed_seq.gravy(),
        "Aromaticity": analysed_seq.aromaticity(),
        "Instability": analysed_seq.instability_index()
    }

if os.path.exists(PHYSCHEM_PATH):
    print("Fizicko-hemijske osobine vec postoje, ucitavanje...")
    physchem_df = pd.read_csv(PHYSCHEM_PATH, index_col="Entry")
else:
    print("Racunam fizicko-hemijske osobine...")
    physchem_df = dataset["Sequence"].apply(lambda seq: pd.Series(extract_physicochemical(seq)))
    physchem_df.index = dataset["Entry"]
    physchem_df.to_csv(PHYSCHEM_PATH)
    print(f"Sacuvano u '{PHYSCHEM_PATH}'")

physchem_df.head()

Fizicko-hemijske osobine vec postoje, ucitavanje...


,MW,pI,GRAVY,Aromaticity,Instability
Entry,,,,,
A0A0C4DH62,1910.0464,5.241828,-0.170588,0.176471,2.958824
A0A0A0MT87,1737.9955,9.994937,0.231250,0.125000,21.512500
P01880,47499.2162,7.251087,-0.430465,0.076744,64.091628
P01876,42848.0149,5.426726,-0.210302,0.075377,58.911332
P01877,42333.4538,5.417347,-0.231714,0.074169,57.405396


### Priprema SC podskupa

In [5]:
dataset_sc = dataset[dataset["label_list"].apply(len) == 1].copy()
dataset_sc["label_sc"] = dataset_sc["label_list"].apply(lambda x: x[0])

print(f"Broj proteina u SC podskupu {len(dataset_sc)}")
print(dataset_sc["label_sc"].value_counts())

Broj proteina u SC podskupu 10609
label_sc
Nucleus          3083
Cell membrane    2700
Cytoplasm        2480
Secreted         1453
Mitochondrion     893
Name: count, dtype: int64


#### Podjela na ulazne i ciljne atribute

In [6]:
X_entries = dataset_sc["Entry"].values
y_raw = dataset_sc["label_sc"].values

print(f"Broj sekvenci: {len(X_entries)}")
print(f"Broj oznaka: {len(y_raw)}")

Broj sekvenci: 10609
Broj oznaka: 10609


#### Kodiranje ciljne varijable (Label Encoding)

In [7]:
le = LabelEncoder()
y_encoded = le.fit_transform(y_raw)

print("Mapiranje klasa:")
for num, label in enumerate(le.classes_):
    print(f"{num} -> {label}")

joblib.dump(le, os.path.join(FEATURES_DIR, "sc_label_encoder.pkl"))

Mapiranje klasa:
0 -> Cell membrane
1 -> Cytoplasm
2 -> Mitochondrion
3 -> Nucleus
4 -> Secreted


['../../data/features/localization\\sc_label_encoder.pkl']

#### Train/test split

In [8]:
RANDOM_STATE = 42
TEST_SIZE = 0.2

entries_sc_train, entries_sc_test, y_sc_train, y_sc_test = train_test_split(
    X_entries, y_encoded,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_encoded,
)

print(f"Train skup: {len(entries_sc_train)} proteina (80%)")
print(f"Test skup:  {len(entries_sc_test)} proteina (20%)")

# Čuvamo Entry liste i encoded labele zajedno — jedan izvor (source of truth) za split
pd.DataFrame({"Entry": entries_sc_train, "label_encoded": y_sc_train}).to_csv(
    os.path.join(FEATURES_DIR, "sc_train_entries.csv"), index=False
)
pd.DataFrame({"Entry": entries_sc_test, "label_encoded": y_sc_test}).to_csv(
    os.path.join(FEATURES_DIR, "sc_test_entries.csv"), index=False
)

Train skup: 8487 proteina (80%)
Test skup:  2122 proteina (20%)


### Priprema MC podskupa

In [9]:
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

mlb = MultiLabelBinarizer()
y_mc = mlb.fit_transform(dataset["label_list"])
label_columns = mlb.classes_

joblib.dump(mlb, os.path.join(FEATURES_DIR, "mc_label_binarizer.pkl"))

mc_label_df = pd.DataFrame(y_mc, columns=label_columns, index=dataset["Entry"])
mc_label_df.to_csv(os.path.join(FEATURES_DIR, "mc_labels.csv"))

splitter = MultilabelStratifiedShuffleSplit(
    n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

train_idx, test_idx = next(splitter.split(dataset["Entry"], y_mc))

entries_mc_train = dataset["Entry"].iloc[train_idx]
entries_mc_test = dataset["Entry"].iloc[test_idx]

print(f"Train skup: {len(entries_mc_train)} proteina (80%)")
print(f"Test skup:  {len(entries_mc_test)} proteina (20%)")

pd.Series(entries_mc_train).to_csv(os.path.join(FEATURES_DIR, "mc_train_entries.csv"), index=False)
pd.Series(entries_mc_test).to_csv(os.path.join(FEATURES_DIR, "mc_test_entries.csv"), index=False)


Train skup: 11708 proteina (80%)
Test skup:  2858 proteina (20%)


### AAC CountVectorizer

In [10]:
AMINO_ACIDS = ['A','C','D','E','F','G','H','I','K','L',
               'M','N','P','Q','R','S','T','V','W','Y']
cv_aac = CountVectorizer(analyzer='char', vocabulary=AMINO_ACIDS, lowercase=False)
normalizer = Normalizer(norm='l1')

aac_matrix = normalizer.transform(cv_aac.fit_transform(dataset["Sequence"]))
aac_df = pd.DataFrame(aac_matrix.toarray(), columns=AMINO_ACIDS, index=dataset["Entry"])

aac_df.to_csv(os.path.join(FEATURES_DIR, "aac_cv_features.csv"))
print(f"AAC-CV dimenzije: {aac_df.shape}")
aac_df.head()

AAC-CV dimenzije: (14566, 20)


,A,C,D,E,F,G,H,I,K,L,M,N,P,Q,R,S,T,V,W,Y
Entry,,,,,,,,,,,,,,,,,,,,
A0A0C4DH62,0.058824,0.000000,0.000000,0.058824,0.058824,0.117647,0.058824,0.000000,0.000000,0.058824,0.000000,0.000000,0.000000,0.117647,0.000000,0.117647,0.117647,0.117647,0.058824,0.058824
A0A0A0MT87,0.125000,0.000000,0.000000,0.000000,0.062500,0.125000,0.000000,0.062500,0.062500,0.125000,0.000000,0.062500,0.000000,0.062500,0.062500,0.062500,0.062500,0.062500,0.000000,0.062500
P01880,0.074419,0.018605,0.034884,0.062791,0.027907,0.046512,0.020930,0.023256,0.048837,0.097674,0.011628,0.023256,0.095349,0.053488,0.048837,0.100000,0.097674,0.065116,0.025581,0.023256
P01876,0.062814,0.035176,0.032663,0.052764,0.035176,0.060302,0.017588,0.010050,0.030151,0.115578,0.005025,0.022613,0.115578,0.042714,0.035176,0.103015,0.118090,0.065327,0.020101,0.020101
P01877,0.066496,0.038363,0.035806,0.056266,0.033248,0.061381,0.020460,0.010230,0.030691,0.112532,0.007673,0.025575,0.104859,0.046036,0.038363,0.092072,0.109974,0.069054,0.020460,0.020460


### Priprema raw sekvenci za TF-IDF (SC i MC)

Izdvajamo i čuvamo sirove sekvence po Entry-ju, poravnate sa train/test splitovima. Fitovanje TF-IDF+SVD se nalazi u notebook-ovima 04 (za SC) i 07 (za MC). 

In [11]:
seq_by_entry = dataset.set_index("Entry")["Sequence"]

# SC raw sekvence
sc_train_seq = seq_by_entry.loc[entries_sc_train]
sc_test_seq = seq_by_entry.loc[entries_sc_test]

sc_train_seq_df = sc_train_seq.rename("Sequence").reset_index()
sc_test_seq_df = sc_test_seq.rename("Sequence").reset_index()

sc_train_seq_df.to_csv(os.path.join(FEATURES_DIR, "sc_train_sequences_raw.csv"), index=False)
sc_test_seq_df.to_csv(os.path.join(FEATURES_DIR, "sc_test_sequences_raw.csv"), index=False)

print(f"SC raw sekvence - train: {len(sc_train_seq_df)} | test: {len(sc_test_seq_df)}")

# MC raw sekvence
mc_train_seq = seq_by_entry.loc[entries_mc_train]
mc_test_seq = seq_by_entry.loc[entries_mc_test]

mc_train_seq_df = mc_train_seq.rename("Sequence").reset_index()
mc_test_seq_df = mc_test_seq.rename("Sequence").reset_index()

mc_train_seq_df.to_csv(os.path.join(FEATURES_DIR, "mc_train_sequences_raw.csv"), index=False)
mc_test_seq_df.to_csv(os.path.join(FEATURES_DIR, "mc_test_sequences_raw.csv"), index=False)

print(f"MC raw sekvence - train: {len(mc_train_seq_df)} | test: {len(mc_test_seq_df)}")

SC raw sekvence - train: 8487 | test: 2122
MC raw sekvence - train: 11708 | test: 2858


### Čuvanje raw (neskaliranih) fizičko-hemijskih osobina (SC i MC)

In [12]:
def save_raw_physchem(train_entries, test_entries, prefix):
    train_ph_raw = physchem_df.loc[train_entries]
    test_ph_raw = physchem_df.loc[test_entries]

    train_ph_raw.to_csv(os.path.join(FEATURES_DIR, f"{prefix}_train_physchem_raw.csv"))
    test_ph_raw.to_csv(os.path.join(FEATURES_DIR, f"{prefix}_test_physchem_raw.csv"))

    return train_ph_raw, test_ph_raw

sc_train_ph_raw, sc_test_ph_raw = save_raw_physchem(entries_sc_train, entries_sc_test, "sc")
mc_train_ph_raw, mc_test_ph_raw = save_raw_physchem(entries_mc_train, entries_mc_test, "mc")

print(f"SC raw PH osobine — train: {sc_train_ph_raw.shape} | test: {sc_test_ph_raw.shape}")
print(f"MC raw PH osobine — train: {mc_train_ph_raw.shape} | test: {mc_test_ph_raw.shape}")

SC raw PH osobine — train: (8487, 5) | test: (2122, 5)
MC raw PH osobine — train: (11708, 5) | test: (2858, 5)


### TF-IDF (Term Frequency-Inverse Document Frequency)

Transformišemo proteinske sekvence u numeričke vektore koristeći **TF-IDF** statistiku. TF-IDF penalizuje n-grame koji se prečesto pojavljuju u svim klasama, a naglašava one koji su jedinstveni za specifične funkcionalne grupe.

In [ ]:
'''def fit_transform_tfidf_svd(train_sequences, test_sequences, n_components=150, ngram_range=(2,4), max_features=5000):
    tfidf = TfidfVectorizer(
        analyzer='char', 
        ngram_range=ngram_range, 
        max_features=max_features, 
        lowercase=False
    )
    
    svd = TruncatedSVD(n_components=n_components, random_state=RANDOM_STATE)

    train_tfidf = tfidf.fit_transform(train_sequences)
    test_tfidf = tfidf.transform(test_sequences)

    train_svd = svd.fit_transform(train_tfidf)
    test_svd = svd.transform(test_tfidf)

    explained_var = svd.explained_variance_ratio_.sum()
    print(f"SVD objašnjena varijansa ({n_components} komponenti): {explained_var:.3f}")

    return train_svd, test_svd, tfidf, svd'''

'def fit_transform_tfidf_svd(train_sequences, test_sequences, n_components=150, ngram_range=(2,4), max_features=5000):\n    tfidf = TfidfVectorizer(\n        analyzer=\'char\', \n        ngram_range=ngram_range, \n        max_features=max_features, \n        lowercase=False\n    )\n    \n    svd = TruncatedSVD(n_components=n_components, random_state=RANDOM_STATE)\n\n    train_tfidf = tfidf.fit_transform(train_sequences)\n    test_tfidf = tfidf.transform(test_sequences)\n\n    train_svd = svd.fit_transform(train_tfidf)\n    test_svd = svd.transform(test_tfidf)\n\n    explained_var = svd.explained_variance_ratio_.sum()\n    print(f"SVD objašnjena varijansa ({n_components} komponenti): {explained_var:.3f}")\n\n    return train_svd, test_svd, tfidf, svd'

### TF-IDF+SVD za Eksperiment A (SC)

In [ ]:
'''seq_by_entry = dataset.set_index("Entry")["Sequence"]

sc_train_seq = seq_by_entry.loc[entries_sc_train]
sc_test_seq = seq_by_entry.loc[entries_sc_test]

sc_train_tfidf, sc_test_tfidf, tfidf_sc, svd_sc = fit_transform_tfidf_svd(sc_train_seq, sc_test_seq)

# Cuvanje matrica
np.save(os.path.join(FEATURES_DIR, "sc_train_tfidf_svd.npy"), sc_train_tfidf)
np.save(os.path.join(FEATURES_DIR, "sc_test_tfidf_svd.npy"), sc_test_tfidf)
# Cuvanje transformera
joblib.dump(tfidf_sc, os.path.join(FEATURES_DIR, "sc_tfidf_vectorizer.pkl"))
joblib.dump(svd_sc, os.path.join(FEATURES_DIR, "sc_svd.pkl"))'''

'seq_by_entry = dataset.set_index("Entry")["Sequence"]\n\nsc_train_seq = seq_by_entry.loc[entries_sc_train]\nsc_test_seq = seq_by_entry.loc[entries_sc_test]\n\nsc_train_tfidf, sc_test_tfidf, tfidf_sc, svd_sc = fit_transform_tfidf_svd(sc_train_seq, sc_test_seq)\n\n# Cuvanje matrica\nnp.save(os.path.join(FEATURES_DIR, "sc_train_tfidf_svd.npy"), sc_train_tfidf)\nnp.save(os.path.join(FEATURES_DIR, "sc_test_tfidf_svd.npy"), sc_test_tfidf)\n# Cuvanje transformera\njoblib.dump(tfidf_sc, os.path.join(FEATURES_DIR, "sc_tfidf_vectorizer.pkl"))\njoblib.dump(svd_sc, os.path.join(FEATURES_DIR, "sc_svd.pkl"))'

### TF-IDF+SVD za Eksperiment C (MC)

In [ ]:
'''mc_train_seq = seq_by_entry.loc[entries_mc_train]
mc_test_seq = seq_by_entry.loc[entries_mc_test]

mc_train_tfidf, mc_test_tfidf, tfidf_mc, svd_mc = fit_transform_tfidf_svd(mc_train_seq, mc_test_seq)

# Cuvanje matrica
np.save(os.path.join(FEATURES_DIR, "mc_train_tfidf_svd.npy"), mc_train_tfidf)
np.save(os.path.join(FEATURES_DIR, "mc_test_tfidf_svd.npy"), mc_test_tfidf)
# Cuvanje transformera
joblib.dump(tfidf_mc, os.path.join(FEATURES_DIR, "mc_tfidf_vectorizer.pkl"))
joblib.dump(svd_mc, os.path.join(FEATURES_DIR, "mc_svd.pkl"))'''

'mc_train_seq = seq_by_entry.loc[entries_mc_train]\nmc_test_seq = seq_by_entry.loc[entries_mc_test]\n\nmc_train_tfidf, mc_test_tfidf, tfidf_mc, svd_mc = fit_transform_tfidf_svd(mc_train_seq, mc_test_seq)\n\n# Cuvanje matrica\nnp.save(os.path.join(FEATURES_DIR, "mc_train_tfidf_svd.npy"), mc_train_tfidf)\nnp.save(os.path.join(FEATURES_DIR, "mc_test_tfidf_svd.npy"), mc_test_tfidf)\n# Cuvanje transformera\njoblib.dump(tfidf_mc, os.path.join(FEATURES_DIR, "mc_tfidf_vectorizer.pkl"))\njoblib.dump(svd_mc, os.path.join(FEATURES_DIR, "mc_svd.pkl"))'

### Skaliranje fizičko-hemijskih osobina

In [ ]:
'''def scale_physchem(train_entries, test_entries, prefix):
    scaler = StandardScaler()
    train_ph = scaler.fit_transform(physchem_df.loc[train_entries])
    test_ph = scaler.transform(physchem_df.loc[test_entries])

    # Cuvanje matrice
    np.save(os.path.join(FEATURES_DIR, f"{prefix}_train_physchem_scaled.npy"), train_ph)
    np.save(os.path.join(FEATURES_DIR, f"{prefix}_test_physchem_scaled.npy"), test_ph)
    # Cuvanje transformera
    joblib.dump(scaler, os.path.join(FEATURES_DIR, f"{prefix}_physchem_scaler.pkl"))

    return train_ph, test_ph

sc_train_ph, sc_test_ph = scale_physchem(entries_sc_train, entries_sc_test, "sc")
mc_train_ph, mc_test_ph = scale_physchem(entries_mc_train, entries_mc_test, "mc")'''

'def scale_physchem(train_entries, test_entries, prefix):\n    scaler = StandardScaler()\n    train_ph = scaler.fit_transform(physchem_df.loc[train_entries])\n    test_ph = scaler.transform(physchem_df.loc[test_entries])\n\n    # Cuvanje matrice\n    np.save(os.path.join(FEATURES_DIR, f"{prefix}_train_physchem_scaled.npy"), train_ph)\n    np.save(os.path.join(FEATURES_DIR, f"{prefix}_test_physchem_scaled.npy"), test_ph)\n    # Cuvanje transformera\n    joblib.dump(scaler, os.path.join(FEATURES_DIR, f"{prefix}_physchem_scaler.pkl"))\n\n    return train_ph, test_ph\n\nsc_train_ph, sc_test_ph = scale_physchem(entries_sc_train, entries_sc_test, "sc")\nmc_train_ph, mc_test_ph = scale_physchem(entries_mc_train, entries_mc_test, "mc")'